In [ ]:
import snscrape.modules.twitter as sntwitter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

In [ ]:
# Allocating tweets to a list

tweets = []
query = "food OR #food OR #recipe lang:en since:2023-01-01 until:2023-12-31"

for i, tweet in enumerate(sntwitter.TwitterSearchScraper(query).get_items()):
    if i > 1000:  # Limit to first 1000 tweets
        break
    tweets.append([tweet.date, tweet.content, tweet.likeCount, tweet.retweetCount, tweet.user.followersCount])

df = pd.DataFrame(tweets, columns=["date", "content", "likes", "retweets", "followers"])

In [ ]:
# Creating popularity labels based on likes

def label_popularity(likes):
    if likes < 50:
        return 0   # low
    elif likes < 200:
        return 1   # medium
    else:
        return 2   # high

df["popularity"] = df["likes"].apply(label_popularity)

In [ ]:
# Preparing Data for Modeling

X = np.array(df[["likes", "retweets", "followers"]])  # Features
Y = np.array(df["popularity"])                        # Labels

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
# Creating and compiling the neural network model

model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(X.shape[1],)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')   # Three classes: low, medium, high
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Training the model

history = model.fit(X, Y, validation_split=0.2, epochs=50, batch_size=32)

In [ ]:
# Evaluating the model

Y_pred = np.argmax(model.predict(X), axis=-1)

print("Accuracy:", accuracy_score(Y, Y_pred))
print("Precision:", precision_score(Y, Y_pred, average='weighted'))
print("Recall:", recall_score(Y, Y_pred, average='weighted'))
print("Classification Report:\n", classification_report(Y, Y_pred))

True

In [ ]:
# Visualizing training history

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(history.history['accuracy'], label='Train Acc', color='yellow')
ax.plot(history.history['val_accuracy'], label='Val Acc', color='orange')
ax.set_title("Accuracy Over Epochs")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(history.history['loss'], label='Train Loss', color='green')
ax.plot(history.history['val_loss'], label='Val Loss', color='red')
ax.set_title("Loss Over Epochs")
ax.legend()
plt.show()

In [ ]:
# Function to predict popularity class for new data

def predict_popularity(likes, retweets, followers):
    # Preparing input data
    x = np.array([[likes, retweets, followers]])
    x = scaler.transform(x)  # Normalization
    
    # Predicting class
    pred = model.predict(x)
    class_idx = np.argmax(pred, axis=-1)[0]
    prob = pred[0][class_idx]
    
    # Mapping class index to label 
    labels = {0: "Low", 1: "Medium", 2: "High"}
    
    return labels[class_idx], prob

In [ ]:
# Testing inference function

tweet1 = predict_popularity(likes=20, retweets=2, followers=150)
tweet2 = predict_popularity(likes=120, retweets=15, followers=5000)
tweet3 = predict_popularity(likes=500, retweets=100, followers=20000)

print("Tweet 1:", tweet1)
print("Tweet 2:", tweet2)
print("Tweet 3:", tweet3)